In [3]:
%pip install kagglehub
%pip install pandas
%pip install scipy
%pip install matplotlib


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pillow-12.1.1-cp310-cp310-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.8-cp310-cp310-win_amd64.whl (8.1 MB)
Using cached contourpy-1.3.2-cp310-cp310-win_amd64.whl (221 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   --------------------------------- ------ 1.3/1.6 MB 7.5 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 6.4 MB/s  0:00:00
Using cached pillow-12.1.1-cp310-cp310-win_amd6

In [29]:
import pandas
df = pandas.read_csv("dataset/games.csv")

In [30]:
# ------- Clean data --------

# Replace true and false if we haven't already
if (df['rated'].dtype == bool):
    mapping = {"rated": {True: 1, False: 0}}

    # Map the labels to values
    for col, m in mapping.items():
        df[col] = df[col].map(m)

# Use one hot encoding for the following features
one_hot_encoded = pandas.get_dummies(df, columns=['victory_status', 'opening_name', 'winner'], drop_first=True, dtype=int)
one_hot_encoded.to_csv('dataset/clean_games.csv', index=False)

In [ ]:
# ---------- EDA - How do the features affect rating? -----------
import os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import json

df = pandas.read_csv("dataset/clean_games.csv")

output_dir = "EDA"
os.makedirs(output_dir, exist_ok=True)

# Features to plot a scatterplot for
feature_plot = ['rated', 'created_at', 'last_move_at', 'turns', 'increment_code', 'black_rating', 'opening_ply']

# Scatterplot
for i in range(1, len(df.columns)):
    if df.columns[i] in feature_plot:
        filename = f'{df.columns[i]}_vs_white_rating_scatter'
        filepath = os.path.join(output_dir, filename)

        plt.figure(figsize=(12, 6))
        plt.scatter(df[df.columns[i]], df['white_rating'])
        plt.ylabel('WHITE RATING')
        plt.xlabel(df.columns[i].upper())
        plt.savefig(filepath)
        plt.close()

# Features to calculate pcc for
feature_pcc = ['rated', 'created_at', 'last_move_at', 'turns', 'black_rating', 'opening_ply']

# Format of dictionary will be:   Key = Feature | Value = [correlation coefficient, p_value]
pcc_dict = {}
# Pearson Correlation Coefficient
for column in df.columns:
    if column in feature_pcc:
        corr_co, p_value = pearsonr(df[column], df['white_rating'])
        pcc_dict[column] = (corr_co, p_value)

# Store values in a txt file
with open("EDA/pcc_values.txt", 'w') as file:
    json.dump(pcc_dict, file, indent=4)

rated
created_at
last_move_at
turns
black_rating
opening_ply
